# Random Forest Practice

**Topic:** Random Forests &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(3000, 30)`, `y_train` `(3000,)`: the label depends on interactions among features 0–4 only, and features 5–29 are noise. ~4% of labels are flipped.
- `X_test` `(1000, 30)`: predict these. `y_test` is used by the evaluation cell.

## Your task

1. Fit a `RandomForestClassifier` with enough trees (hundreds) and only light depth controls: the label comes from interactions, and a forest capped at `max_depth=8` or shallower falls short of the target. Compare against a single decision tree if you're curious.
2. Peek at `feature_importances_`: the informative features should stand out. This step is not graded.
3. Set `predictions` to your test predictions.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    X = r.normal(0, 1, (n, 30))
    score = (
        2.2 * X[:, 0] * X[:, 1]           # interaction
        + 1.6 * np.sign(X[:, 2]) * np.abs(X[:, 3])
        - 1.4 * (X[:, 4] > 0.5) * X[:, 0]
    )
    y = (score > 0).astype(int)
    flip = r.random(n) < 0.04
    return X, np.where(flip, 1 - y, y)

X_train, y_train = make(3000, 81)
X_test, y_test = make(1000, 82)
print(f"X_train {X_train.shape}, label balance {y_train.mean():.2f}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.78

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

Hundreds of bagged trees with feature subsampling average away the label noise, and `min_samples_leaf=3` is a light depth control that leaves the trees deep enough to capture the interactions among features 0-4. This forest scores about 0.81 on the test set. With 25 distractor features among 30, the default `max_features="sqrt"` offers each split only 5 columns, and more than a third of those draws contain no informative feature. Passing `max_features=10` lifts the score to about 0.87.

```python
import numpy as np
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(
    n_estimators=400, min_samples_leaf=3, random_state=0, n_jobs=-1
)
forest.fit(X_train, y_train)
print("top importances:", np.argsort(forest.feature_importances_)[::-1][:5])
predictions = forest.predict(X_test)
```
